In [11]:
itens = [
'ALOS3',
'ALPA4',
'ABEV3',
'ANIM3',
'ASAI3',
'AURE3',
'AXIA3',
'AZZA3',
'B3SA3',
'BBSE3',
'BBDC3',
'BBDC4',
'BRAP4',
'SAUD3',
'BBAS3',
'BRAV3',
'BPAC11',
'CXSE3',
'CBAV3',
'CEAB3',
'CMIG4',
'COGN3',
'CSMG3',
'CPLE3',
'CSAN3',
'CPFE3',
'CMIN3',
'CURY3',
'CVCB3',
'CYRE3',
'DIRR3',
'ECOR3',
'EMBJ3',
'ENGI11',
'ENEV3',
'EGIE3',
'EQTL3',
'EZTC3',
'FLRY3',
'GGBR4',
'GOAU4',
'GGPS3',
'GMAT3',
'SBFG3',
'HAPV3',
'HYPE3',
'IGTI11',
'INTB3',
'IRBR3',
'ISAE4',
'ITSA4',
'ITUB3',
'ITUB4',
'JHSF3',
'KLBN11',
'RENT3',
'LREN3',
'MGLU3',
'POMO4',
'MBRF3',
'BEEF3',
'MOTV3',
'MDNE3',
'MOVI3',
'MRVE3',
'MULT3',
'NATU3',
'ORVR3',
'PGMN3',
'PETR3',
'PETR4',
'RECV3',
'PLPL3',
'PSSA3',
'PRIO3',
'RADL3',
'RDOR3',
'RAIL3',
'SBSP3',
'SAPR11',
'SANB11',
'SMTO3',
'CSNA3',
'SIMH3',
'SLCE3',
'SMFT3',
'SUZB3',
'TAEE11',
'VIVT3',
'TEND3',
'TIMS3',
'TOTS3',
'UGPA3',
'USIM5',
'VALE3',
'VAMO3',
'VBBR3',
'VIVA3',
'WEGE3',
'YDUQ3',

]

In [12]:
import os, time
import pandas as pd
import numpy as np
import yfinance as yf

# ---------- tickers ----------
# mesma origem de antes; ajuste se os arquivos tiverem extensão
tickers = sorted({os.path.splitext(f)[0].upper().strip() for f in itens})
print(f"{len(tickers)} tickers: {tickers[:8]} ...")

INICIO, FIM = '2015-01-01', '2026-07-21'   # FIM é exclusivo

# ---------- download em blocos ----------
def baixar(simbolos, inicio, fim, tamanho=30, pausa=1.0):
    partes = []
    for i in range(0, len(simbolos), tamanho):
        bloco = simbolos[i:i+tamanho]
        d = yf.download(bloco, start=inicio, end=fim, interval='1d',
                        auto_adjust=True, actions=False,
                        progress=False, group_by='column')
        if isinstance(d.columns, pd.MultiIndex):
            d = d['Close']
        else:                                  # bloco de 1 ticker
            d = d[['Close']].rename(columns={'Close': bloco[0]})
        partes.append(d)
        print(f"  bloco {i//tamanho+1}: {d.shape[1]} colunas")
        time.sleep(pausa)
    return pd.concat(partes, axis=1)

simbolos = [t + '.SA' for t in tickers]
precos = baixar(simbolos, INICIO, FIM)

# ---------- limpeza ----------
precos.columns = [c.replace('.SA', '') for c in precos.columns]
precos.index = pd.to_datetime(precos.index).tz_localize(None).normalize()
precos = precos.sort_index()
precos = precos[~precos.index.duplicated(keep='last')]
precos = precos.reindex(columns=tickers)        # ordem estável

# tickers que o Yahoo não conhece voltam só com NaN
vazios = precos.columns[precos.isna().all()].tolist()
if vazios:
    print(f"\n!! SEM DADOS ({len(vazios)}): {vazios}")
    precos = precos.drop(columns=vazios)

# ---------- retornos ----------
retornos = precos.pct_change(fill_method=None).dropna(how='all')

os.makedirs('retornos', exist_ok=True)
precos.to_csv('retornos/precos_ajustados.csv', index_label='date')
retornos.to_csv('retornos/retornos.csv', index_label='date')

print(f"\nprecos  : {precos.shape}  {precos.index.min().date()} -> {precos.index.max().date()}")
print(f"retornos: {retornos.shape}")

100 tickers: ['ABEV3', 'ALOS3', 'ALPA4', 'ANIM3', 'ASAI3', 'AURE3', 'AXIA3', 'AZZA3'] ...
  bloco 1: 30 colunas
  bloco 2: 30 colunas
  bloco 3: 30 colunas
  bloco 4: 10 colunas

precos  : (2873, 100)  2015-01-02 -> 2026-07-20
retornos: (2872, 100)


In [13]:
# ---------- 1. calendário de eventos societários ----------
eventos = []
for t in retornos.columns:
    try:
        s = yf.Ticker(t + '.SA').splits
        for dt, razao in s.items():
            eventos.append({'ticker': t,
                            'data': pd.Timestamp(dt).tz_localize(None).normalize(),
                            'razao': float(razao)})
        time.sleep(0.2)
    except Exception as e:
        print(f"  splits falhou em {t}: {e}")

ev = pd.DataFrame(eventos)
ev = ev[(ev['data'] >= retornos.index.min()) & (ev['data'] <= retornos.index.max())]

# retorno falso que apareceria se o ajuste tivesse falhado
ev['falso_esperado'] = 1/ev['razao'] - 1
ev['retorno_na_base'] = [retornos[r.ticker].get(r.data, np.nan) for r in ev.itertuples()]
ev['suspeito'] = (ev['retorno_na_base'] - ev['falso_esperado']).abs() < 0.02

print(f"\n{len(ev)} eventos no período, {ev['suspeito'].sum()} suspeitos de NÃO terem sido ajustados")
print(ev.sort_values('data').to_string(index=False))

# ---------- 2. movimentos extremos ----------
s = retornos.abs().stack()
print("\n--- maiores |retorno| da base ---")
print(s.sort_values(ascending=False).head(25).to_string())

# ---------- 3. buracos ----------
falta = retornos.isna().sum().sort_values(ascending=False)
print(f"\n--- tickers com mais NaN (de {len(retornos)} pregões) ---")
print(falta[falta > 0].head(20).to_string())


124 eventos no período, 3 suspeitos de NÃO terem sido ajustados
ticker       data     razao  falso_esperado  retorno_na_base  suspeito
 BBDC3 2015-03-27  1.200000       -0.166667        -0.037473     False
 BBDC4 2015-03-27  1.200000       -0.166667        -0.027762     False
 WEGE3 2015-04-01  2.000000       -0.500000         0.016981     False
 CPFE3 2015-04-30  1.031945       -0.030956        -0.019924      True
 EZTC3 2015-05-04  1.070434       -0.065799        -0.007023     False
 ITSA4 2015-05-05  1.100000       -0.090909        -0.011951     False
 ITUB3 2015-07-14  1.100000       -0.090909        -0.001294     False
 ITUB4 2015-07-14  1.100000       -0.090909         0.000685     False
 RAIL3 2015-08-03  0.100000        9.000000        -0.027472     False
 LREN3 2015-09-24  5.000000       -0.800000        -0.014356     False
 MGLU3 2015-10-01  0.125000        7.000000        -0.086807     False
 BBDC4 2016-04-18  1.100000       -0.090909        -0.005689     False
 BBDC3 2016-

In [14]:
LIM_NAN = 0.05          # descarta ativo com mais de 5% de pregões faltando no bloco

# os.makedirs('retornos/anos', exist_ok=True)
dfs = {}
for ano in range(2015, 2026):
    ini = pd.Timestamp(f'{ano}-04-01')
    fim = pd.Timestamp(f'{ano+1}-03-31')
    bloco = retornos.loc[ini:fim].copy()

    ok = bloco.columns[(bloco.isna().mean() <= LIM_NAN) & (bloco.fillna(0) != 0).any()]
    bloco = bloco[ok].fillna(0.0)      # NaN residual -> dia sem variação

    dfs[str(ano)] = bloco
    bloco.to_csv(f'plotagens/df_acumulados_{ano}.csv', index_label='date')
    print(f"{ano}: {bloco.shape[0]:>3} pregões, {bloco.shape[1]:>3} ativos  "
          f"({bloco.index.min().date()} -> {bloco.index.max().date()})")

2015: 245 pregões,  70 ativos  (2015-04-01 -> 2016-03-31)
2016: 251 pregões,  70 ativos  (2016-04-01 -> 2017-03-31)
2017: 253 pregões,  71 ativos  (2017-04-03 -> 2018-03-29)
2018: 245 pregões,  76 ativos  (2018-04-02 -> 2019-03-29)
2019: 250 pregões,  76 ativos  (2019-04-01 -> 2020-03-31)
2020: 246 pregões,  80 ativos  (2020-04-01 -> 2021-03-31)
2021: 249 pregões,  92 ativos  (2021-04-01 -> 2022-03-31)
2022: 251 pregões,  99 ativos  (2022-04-01 -> 2023-03-31)
2023: 246 pregões,  99 ativos  (2023-04-03 -> 2024-03-28)
2024: 251 pregões,  99 ativos  (2024-04-01 -> 2025-03-31)
2025: 250 pregões,  99 ativos  (2025-04-01 -> 2026-03-31)


In [20]:
LIM_NAN = 0.05
# DIR_IN  = '/dados_retornos/otimizacao/ativos'      # dentro da amostra
os.makedirs(DIR_IN, exist_ok=True)


dfs_in = {}
for ano in range(2015, 2026):                        # 2015 -> carteira de 2016
    ini = pd.Timestamp(f'{ano}-10-01')
    fim = pd.Timestamp(f'{ano+1}-03-31')
    bloco = retornos.loc[ini:fim].copy()
    ok = bloco.columns[(bloco.isna().mean() <= LIM_NAN) & (bloco.fillna(0) != 0).any()]
    bloco = bloco[ok].fillna(0.0)
    print(len(bloco.columns))

    dfs_in[str(ano)] = bloco
    bloco.to_csv(f'dados_retornos/otimizacao/ativos/df_ativos_{ano}.csv', index_label='date')
    print(f"{ano}: {bloco.shape[0]:>3} pregões, {bloco.shape[1]:>3} ativos  "
          f"({bloco.index.min().date()} -> {bloco.index.max().date()})")

70
2015: 120 pregões,  70 ativos  (2015-10-01 -> 2016-03-31)
69
2016: 123 pregões,  69 ativos  (2016-10-03 -> 2017-03-31)
73
2017: 126 pregões,  73 ativos  (2017-10-02 -> 2018-03-29)
76
2018: 119 pregões,  76 ativos  (2018-10-01 -> 2019-03-29)
77
2019: 123 pregões,  77 ativos  (2019-10-01 -> 2020-03-31)
84
2020: 120 pregões,  84 ativos  (2020-10-01 -> 2021-03-31)
97
2021: 123 pregões,  97 ativos  (2021-10-01 -> 2022-03-31)
99
2022: 124 pregões,  99 ativos  (2022-10-03 -> 2023-03-31)
99
2023: 121 pregões,  99 ativos  (2023-10-02 -> 2024-03-28)
99
2024: 122 pregões,  99 ativos  (2024-10-01 -> 2025-03-31)
100
2025: 123 pregões, 100 ativos  (2025-10-01 -> 2026-03-31)
